# Beyond Accuracy — Full Pipeline (Single Notebook)

Combines the proven `beyondaccuracy.ipynb` checkpoint/resume pipeline with the
`src/session_init.py` / `src/session_save.py` cross-session persistence layer.

Runs Steps 0-5 end-to-end in one Kaggle session:
- **Step 0** - environment smoke test
- **Step 1** - download GSM8K / BBH / TruthfulQA, generate 5 paraphrases per question
- **Step 2** - accuracy (direct vs CoT) across all 7 models
- **Step 3** - calibration (ECE, Brier, AUROC, AURC) on GSM8K + TruthfulQA
- **Step 4** - paraphrase consistency (CoT vs direct)
- **Step 5** - analysis + all 5 figures + hypothesis checks

**Resume model:** every step is gated by a `checkpoints/stepN.done` marker (from the
proven notebook) AND every result CSV is itself resume-safe via `done_keys`
(model, mode, dataset) skip-logic — so even a step interrupted halfway resumes
mid-step, not just at step boundaries.

**Optional Kaggle datasets to attach:**
- `light-paper` (or similarly named) - restores `light_paper/` incl. `results/`, `data/`, `checkpoints/` from a previous session
- Without it attached, the notebook clones `BeyondAccuracy` from GitHub and starts fresh.

## Restore or clone the resumable project

In [ ]:
import os
import shutil
import sys
from pathlib import Path

WORK_DIR  = Path("/kaggle/working/light_paper")
INPUT_DIR = Path("/kaggle/input/datasets/abhivirsingh/beyondaccuracy-test/light_paper")
REPO_URL  = "https://github.com/bharathgaddam1712/BeyondAccuracy.git"


def find_dataset_project(dataset_dir: Path):
    """Support both a flat dataset and a dataset containing light_paper/."""
    for candidate in [dataset_dir / "light_paper", dataset_dir]:
        if (candidate / "src").is_dir():
            return candidate
    return None


dataset_project = find_dataset_project(INPUT_DIR) if INPUT_DIR.exists() else None
working_project = find_dataset_project(WORK_DIR) if WORK_DIR.exists() else None

if working_project is not None:
    print(f"Using existing working copy: {working_project}")
elif dataset_project is not None:
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copytree(dataset_project, WORK_DIR, dirs_exist_ok=True)
    print(f"Restored complete project from {dataset_project} -> {WORK_DIR}")
else:
    if WORK_DIR.exists():
        shutil.rmtree(WORK_DIR)
    WORK_DIR.parent.mkdir(parents=True, exist_ok=True)
    get_ipython().system(f"git clone {REPO_URL} {WORK_DIR}")
    print(f"Cloned repository into {WORK_DIR}")

os.chdir(WORK_DIR)
ROOT = WORK_DIR
sys.path.insert(0, str(WORK_DIR))

CHECKPOINT_DIR = WORK_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


def checkpoint_path(step: int) -> Path:
    return CHECKPOINT_DIR / f"step{step}.done"


def stage_done(step: int) -> bool:
    return checkpoint_path(step).exists()


def mark_stage_done(step: int) -> None:
    checkpoint_path(step).write_text("completed\n")
    print(f"checkpoint saved: {checkpoint_path(step)}")


for directory in ["data", "results", "figures", "outputs"]:
    (WORK_DIR / directory).mkdir(parents=True, exist_ok=True)

print(f"Working directory: {Path.cwd()}")
print(f"Attached dataset found: {dataset_project is not None}")
print(f"Completed stages: {[step for step in range(6) if stage_done(step)]}")


## Install runtime dependencies

In [ ]:
# Kaggle sessions are fresh after a timeout, so install dependencies on every
# session before importing project modules.
get_ipython().system("pip install -q transformers==4.44.0 accelerate bitsandbytes datasets evaluate")
get_ipython().system("pip install -q pandas numpy matplotlib seaborn scipy scikit-learn")
print("Packages installed")


## Optional Hugging Face authentication

In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login

    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = hf_token
    login(token=hf_token)
    print("Authenticated with Hugging Face")
except Exception as exc:
    print(f"HF_TOKEN unavailable; continuing unauthenticated: {exc}")


## GPU check

In [ ]:
import torch

print(f"PyTorch CUDA available: {torch.cuda.is_available()}")
print(f"Device count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name}, {props.total_memory / 1e9:.1f} GB")


## Session init (cross-session persistence)

Runs `src/session_init.py::init()` on top of the restored/cloned project.
This is a no-op safety net beyond the restore cell above: if `slm-cot-results`
or `slm-cot-models` Kaggle datasets are separately attached, it also restores
any results CSVs and points the HF cache at pre-downloaded model weights —
independent of whether the main `light-paper` project dataset was attached.

In [ ]:
from src.session_init import init

P = init(base_dir=WORK_DIR)
print("Paths:", P)


## STEP 0 — Environment smoke test

In [ ]:
if stage_done(0):
    print("STEP 0 already complete - skipping smoke test")
else:
    from src.models import get_model_by_name, load_model
    from src.prompts import make_accuracy_prompt
    from src.inference import generate_responses

    config = get_model_by_name("Qwen2.5-0.5B")
    model, tokenizer = load_model(config)
    test_prompt = make_accuracy_prompt("What is 2 + 2?", "direct", "gsm8k")
    responses = generate_responses(model, tokenizer, [test_prompt], max_new_tokens=64, batch_size=1)
    print(f"Test response: {responses[0]}")

    import gc
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print("Smoke test passed; GPU freed")
    mark_stage_done(0)


## STEP 1 — Datasets and paraphrases

In [ ]:
if stage_done(1):
    print("STEP 1 already complete - skipping dataset/paraphrase generation")
else:
    import gc
    import json
    import re
    import torch

    from src.datasets import load_all_datasets, save_paraphrases
    from src.inference import generate_responses
    from src.models import get_model_by_name, load_model
    from src.prompts import make_paraphrase_gen_prompt

    DATA_DIR = ROOT / "data"
    DATASETS_PATH = DATA_DIR / "datasets.json"
    PARAPHRASES_PATH = DATA_DIR / "paraphrases.json"

    if DATASETS_PATH.exists():
        with open(DATASETS_PATH) as f:
            all_data = json.load(f)
        print(f"Loaded existing datasets from {DATASETS_PATH}")
    else:
        all_data = load_all_datasets(n_gsm8k=200, n_bbh_per_subtask=60, n_truthfulqa=200)
        with open(DATASETS_PATH, "w") as f:
            json.dump(all_data, f, indent=2)
        print(f"Saved datasets to {DATASETS_PATH}")

    for ds_name, samples in all_data.items():
        print(f"  {ds_name}: {len(samples)} samples")

    def parse_paraphrases(raw: str) -> list:
        lines = [line.strip() for line in raw.split("\n") if line.strip()]
        return [match.group(1).strip() for line in lines
                if (match := re.match(r"^\d+[.)\s]+(.+)", line))][:5]

    if PARAPHRASES_PATH.exists():
        with open(PARAPHRASES_PATH) as f:
            paraphrases = json.load(f)
        print(f"Loaded existing paraphrases: {len(paraphrases)} entries")
    else:
        para_questions = all_data["gsm8k"] + all_data["truthfulqa"]
        para_config = get_model_by_name("Qwen2.5-3B")
        para_model, para_tokenizer = load_model(para_config)
        prompts = [make_paraphrase_gen_prompt(q["question"]) for q in para_questions]
        raw_outputs = generate_responses(para_model, para_tokenizer, prompts, max_new_tokens=512, batch_size=4)
        paraphrases = {q["id"]: parse_paraphrases(raw) for q, raw in zip(para_questions, raw_outputs)}
        save_paraphrases(paraphrases, PARAPHRASES_PATH)
        del para_model, para_tokenizer
        gc.collect()
        torch.cuda.empty_cache()
        print(f"Generated and saved {len(paraphrases)} paraphrase sets")

    mark_stage_done(1)


## STEP 2 — Accuracy (direct vs CoT)

In [ ]:
if stage_done(2):
    print("STEP 2 already complete - skipping")
else:
    import json, gc
    import torch
    import pandas as pd

    BASE_DIR    = ROOT
    DATA_DIR    = BASE_DIR / "data"
    RESULTS_DIR = BASE_DIR / "results"
    RESULTS_DIR.mkdir(exist_ok=True)

    from src.models    import MODEL_REGISTRY, load_model
    from src.prompts   import make_accuracy_prompt
    from src.inference import generate_responses, parse_gsm8k_answer, parse_bbh_answer, parse_truthfulqa_answer
    from src.metrics   import compute_accuracy_by_dataset

    with open(DATA_DIR / "datasets.json") as f:
        all_data = json.load(f)

    MODES = ["direct", "cot"]
    PARSERS = {
        "gsm8k":      parse_gsm8k_answer,
        "bbh":        parse_bbh_answer,
        "truthfulqa": parse_truthfulqa_answer,
    }

    print("Datasets loaded:")
    for k, v in all_data.items():
        print(f"  {k}: {len(v)} samples")

    RAW_PATH = RESULTS_DIR / "accuracy_raw.csv"

    if RAW_PATH.exists():
        existing_df = pd.read_csv(RAW_PATH, keep_default_na=False)
        existing_df["predicted"] = existing_df["predicted"].fillna("").astype(str)
        existing_df["answer"] = existing_df["answer"].fillna("").astype(str)
        done_keys = set(zip(existing_df["model"], existing_df["mode"], existing_df["dataset"]))
        print(f"Resuming - {len(done_keys)} (model, mode, dataset) combos already done")
    else:
        existing_df = pd.DataFrame()
        done_keys = set()

    all_rows = existing_df.to_dict("records") if not existing_df.empty else []

    for model_config in MODEL_REGISTRY:
        for mode in MODES:
            pending_datasets = [ds for ds in ["gsm8k", "bbh", "truthfulqa"] if (model_config.name, mode, ds) not in done_keys]
            if not pending_datasets:
                print(f"Skip {model_config.name}/{mode} (all done)")
                continue

            print(f"\n-- {model_config.name} | mode={mode} --")
            model, tokenizer = load_model(model_config)

            for ds_name in pending_datasets:
                samples = all_data[ds_name]
                parser  = PARSERS[ds_name]

                prompts = [make_accuracy_prompt(s["question"], mode, ds_name, s.get("choices")) for s in samples]
                print(f"  [{ds_name}] {len(prompts)} prompts...")

                responses = generate_responses(model, tokenizer, prompts, max_new_tokens=512, batch_size=4)

                for s, resp in zip(samples, responses):
                    predicted = parser(resp)
                    all_rows.append({
                        "model":        model_config.name,
                        "size_b":       model_config.size_b,
                        "mode":         mode,
                        "dataset":      ds_name,
                        "id":           s["id"],
                        "question":     s["question"][:200],
                        "answer":       s["answer"],
                        "predicted":    predicted if predicted else "",
                        "raw_response": resp[:500],
                    })

                df = pd.DataFrame(all_rows)
                df.to_csv(RAW_PATH, index=False)
                done_keys.add((model_config.name, mode, ds_name))

                subset = df[(df['model'] == model_config.name) & (df['mode'] == mode) & (df['dataset'] == ds_name)]
                acc = (subset["predicted"].str.lower().str.strip() == subset["answer"].str.lower().str.strip()).mean()
                print(f"  [{ds_name}] acc = {acc:.3f}")

            del model, tokenizer
            gc.collect()
            torch.cuda.empty_cache()
            print("  GPU freed")

    df = pd.read_csv(RAW_PATH, keep_default_na=False)
    df["predicted"] = df["predicted"].fillna("").astype(str)
    df["answer"] = df["answer"].fillna("").astype(str)
    df["correct"] = (df["predicted"].str.lower().str.strip() == df["answer"].str.lower().str.strip())

    summary = compute_accuracy_by_dataset(df)
    summary_path = RESULTS_DIR / "accuracy_summary.csv"
    summary.to_csv(summary_path, index=False)

    print(summary.pivot_table(index=["model", "size_b"], columns=["mode", "dataset"], values="accuracy").round(3).to_string())
    print("\nSTEP 2 COMPLETE")
    print("  Outputs: results/accuracy_raw.csv, results/accuracy_summary.csv")
    mark_stage_done(2)


## STEP 3 — Calibration (ECE, Brier, AUROC, AURC)

In [ ]:
if stage_done(3):
    print("STEP 3 already complete - skipping")
else:
    import json, gc
    import torch
    import pandas as pd

    BASE_DIR    = ROOT
    DATA_DIR    = BASE_DIR / "data"
    RESULTS_DIR = BASE_DIR / "results"

    from src.models    import MODEL_REGISTRY, load_model
    from src.prompts   import make_calibration_prompt
    from src.inference import generate_responses, parse_gsm8k_answer, parse_truthfulqa_answer, parse_confidence
    from src.metrics   import compute_ece, compute_calibration_by_model

    with open(DATA_DIR / "datasets.json") as f:
        all_data = json.load(f)

    # Calibration only on GSM8K + TruthfulQA (BBH answers are too short for verbalized confidence)
    CAL_DATASETS = ["gsm8k", "truthfulqa"]
    MODES        = ["direct", "cot"]
    PARSERS = {"gsm8k": parse_gsm8k_answer, "truthfulqa": parse_truthfulqa_answer}

    RAW_PATH = RESULTS_DIR / "calibration_raw.csv"

    if RAW_PATH.exists():
        existing_df = pd.read_csv(RAW_PATH)
        done_keys = set(zip(existing_df["model"], existing_df["mode"], existing_df["dataset"]))
        print(f"Resuming - {len(done_keys)} combos done")
    else:
        existing_df = pd.DataFrame()
        done_keys = set()

    all_rows = existing_df.to_dict("records") if not existing_df.empty else []

    for model_config in MODEL_REGISTRY:
        for mode in MODES:
            pending = [ds for ds in CAL_DATASETS if (model_config.name, mode, ds) not in done_keys]
            if not pending:
                print(f"Skip {model_config.name}/{mode}")
                continue

            print(f"\n-- {model_config.name} | {mode} --")
            model, tokenizer = load_model(model_config)

            for ds_name in pending:
                samples = all_data[ds_name]
                parser  = PARSERS[ds_name]

                prompts = [make_calibration_prompt(s["question"], mode, ds_name, s.get("choices")) for s in samples]
                print(f"  [{ds_name}] {len(prompts)} prompts...")

                responses = generate_responses(model, tokenizer, prompts, max_new_tokens=640, batch_size=4)

                for s, resp in zip(samples, responses):
                    predicted  = parser(resp) or ""
                    confidence = parse_confidence(resp)
                    correct    = (str(predicted).lower().strip() == str(s["answer"]).lower().strip())
                    all_rows.append({
                        "model":        model_config.name,
                        "size_b":       model_config.size_b,
                        "mode":         mode,
                        "dataset":      ds_name,
                        "id":           s["id"],
                        "answer":       s["answer"],
                        "predicted":    predicted,
                        "confidence":   confidence,
                        "correct":      correct,
                        "raw_response": resp[:600],
                    })

                pd.DataFrame(all_rows).to_csv(RAW_PATH, index=False)
                done_keys.add((model_config.name, mode, ds_name))

                valid = [r for r in all_rows if r["model"] == model_config.name and r["mode"] == mode
                         and r["dataset"] == ds_name and r["confidence"] is not None]
                if valid:
                    ece_res = compute_ece([r["confidence"] for r in valid], [r["correct"] for r in valid])
                    conf_avg = sum(r["confidence"] for r in valid) / len(valid)
                    print(f"  [{ds_name}] ECE={ece_res['ece']:.3f}, mean_conf={conf_avg:.3f}, n_valid={len(valid)}")

            del model, tokenizer
            gc.collect()
            torch.cuda.empty_cache()
            print("  GPU freed")

    df = pd.read_csv(RAW_PATH)
    df["correct"] = df["correct"].astype(bool)

    ece_summary = compute_calibration_by_model(df)
    ece_summary.to_csv(RESULTS_DIR / "calibration_ece.csv", index=False)

    bin_data = {}
    for (model, mode, dataset), group in df.groupby(["model", "mode", "dataset"]):
        valid = group.dropna(subset=["confidence"])
        if len(valid) >= 10:
            result = compute_ece(valid["confidence"].tolist(), valid["correct"].tolist())
            bin_data[f"{model}/{mode}/{dataset}"] = result["bin_data"]

    with open(RESULTS_DIR / "calibration_bin_data.json", "w") as f:
        json.dump(bin_data, f, indent=2)

    print("ECE Summary (CoT vs Direct):")
    pivot = ece_summary.pivot_table(index=["model"], columns=["mode", "dataset"], values="ece").round(3)
    print(pivot.to_string())
    print("\nSTEP 3 COMPLETE")
    print("  Outputs: results/calibration_raw.csv, calibration_ece.csv, calibration_bin_data.json")
    mark_stage_done(3)


## STEP 4 — Paraphrase consistency

In [ ]:
if stage_done(4):
    print("STEP 4 already complete - skipping")
else:
    import json, gc
    import torch
    import pandas as pd

    BASE_DIR    = ROOT
    DATA_DIR    = BASE_DIR / "data"
    RESULTS_DIR = BASE_DIR / "results"

    from src.models    import MODEL_REGISTRY, load_model
    from src.prompts   import make_consistency_prompt
    from src.inference import generate_responses, parse_gsm8k_answer, parse_truthfulqa_answer
    from src.metrics   import compute_consistency_by_model, compute_consistency_delta, compute_consistency

    with open(DATA_DIR / "datasets.json") as f:
        all_data = json.load(f)

    with open(DATA_DIR / "paraphrases.json") as f:
        paraphrases = json.load(f)

    CON_DATASETS = ["gsm8k", "truthfulqa"]
    MODES        = ["direct", "cot"]
    PARSERS      = {"gsm8k": parse_gsm8k_answer, "truthfulqa": parse_truthfulqa_answer}

    def expand_with_paraphrases(samples, paraphrases, ds_name):
        """One entry per paraphrase (idx 0 = original question)."""
        expanded = []
        for s in samples:
            qid = s["id"]
            paras = paraphrases.get(qid, [])
            variants = [s["question"]] + paras
            for idx, q in enumerate(variants[:6]):
                expanded.append({
                    "question_id":    qid,
                    "paraphrase_idx": idx,
                    "question":       q,
                    "answer":         s["answer"],
                    "choices":        s.get("choices"),
                    "dataset":        ds_name,
                })
        return expanded

    RAW_PATH = RESULTS_DIR / "consistency_raw.csv"

    if RAW_PATH.exists():
        existing_df = pd.read_csv(RAW_PATH)
        done_keys = set(zip(existing_df["model"], existing_df["mode"], existing_df["dataset"]))
        print(f"Resuming - {len(done_keys)} combos done")
    else:
        existing_df = pd.DataFrame()
        done_keys = set()

    all_rows = existing_df.to_dict("records") if not existing_df.empty else []

    for model_config in MODEL_REGISTRY:
        for mode in MODES:
            pending = [ds for ds in CON_DATASETS if (model_config.name, mode, ds) not in done_keys]
            if not pending:
                print(f"Skip {model_config.name}/{mode}")
                continue

            print(f"\n-- {model_config.name} | {mode} --")
            model, tokenizer = load_model(model_config)

            for ds_name in pending:
                samples  = all_data[ds_name]
                parser   = PARSERS[ds_name]
                expanded = expand_with_paraphrases(samples, paraphrases, ds_name)

                prompts = [make_consistency_prompt(e["question"], mode, ds_name, e.get("choices")) for e in expanded]
                print(f"  [{ds_name}] {len(prompts)} prompts ({len(samples)} questions x ~6 variants)...")

                responses = generate_responses(model, tokenizer, prompts, max_new_tokens=512, batch_size=4)

                for e, resp in zip(expanded, responses):
                    predicted = parser(resp) or ""
                    all_rows.append({
                        "model":          model_config.name,
                        "size_b":         model_config.size_b,
                        "mode":           mode,
                        "dataset":        ds_name,
                        "question_id":    e["question_id"],
                        "paraphrase_idx": e["paraphrase_idx"],
                        "answer":         e["answer"],
                        "predicted":      predicted,
                    })

                pd.DataFrame(all_rows).to_csv(RAW_PATH, index=False)
                done_keys.add((model_config.name, mode, ds_name))

                temp_df = pd.DataFrame(all_rows)
                subset = temp_df[(temp_df['model'] == model_config.name) & (temp_df['mode'] == mode) & (temp_df['dataset'] == ds_name)]
                per_q = subset.groupby("question_id")["predicted"].apply(list)
                mean_con = sum(compute_consistency(a) for a in per_q) / len(per_q) if len(per_q) > 0 else float('nan')
                print(f"  [{ds_name}] mean_consistency = {mean_con:.3f}")

            del model, tokenizer
            gc.collect()
            torch.cuda.empty_cache()
            print("  GPU freed")

    df = pd.read_csv(RAW_PATH)

    summary = compute_consistency_by_model(df)
    summary.to_csv(RESULTS_DIR / "consistency_summary.csv", index=False)

    delta = compute_consistency_delta(summary)
    delta.to_csv(RESULTS_DIR / "consistency_delta.csv", index=False)

    print("Consistency by (model, mode, dataset):")
    pivot = summary.pivot_table(index="model", columns=["mode", "dataset"], values="consistency").round(3)
    print(pivot.to_string())

    print("\nConsistency Delta (CoT - Direct):")
    print(delta[["model", "dataset", "delta"]].to_string(index=False))

    print("\nSTEP 4 COMPLETE")
    mark_stage_done(4)


## STEP 5 — Analysis & Paper Figures

In [ ]:
if stage_done(5):
    print("STEP 5 already complete - skipping")
else:
    import json
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns

    BASE_DIR    = ROOT
    RESULTS_DIR = BASE_DIR / "results"
    FIGURES_DIR = BASE_DIR / "figures"
    FIGURES_DIR.mkdir(exist_ok=True)

    from src.models import MODEL_REGISTRY, MODEL_SIZES, MODEL_NAMES

    plt.rcParams.update({
        "font.size": 11, "axes.titlesize": 12, "axes.labelsize": 11,
        "figure.dpi": 150, "lines.linewidth": 2, "lines.markersize": 7,
    })

    COLORS  = {"direct": "#1f77b4", "cot": "#d62728"}
    MARKERS = {"direct": "o", "cot": "s"}
    DATASETS = ["gsm8k", "bbh", "truthfulqa"]
    DATASET_LABELS = {"gsm8k": "GSM8K", "bbh": "BIG-Bench Hard", "truthfulqa": "TruthfulQA"}

    acc_df   = pd.read_csv(RESULTS_DIR / "accuracy_summary.csv")
    ece_df   = pd.read_csv(RESULTS_DIR / "calibration_ece.csv")
    con_df   = pd.read_csv(RESULTS_DIR / "consistency_summary.csv")
    delta_df = pd.read_csv(RESULTS_DIR / "consistency_delta.csv")

    with open(RESULTS_DIR / "calibration_bin_data.json") as f:
        bin_data = json.load(f)

    size_map = {m.name: m.size_b for m in MODEL_REGISTRY}
    for df in [acc_df, ece_df, con_df, delta_df]:
        if "size_b" not in df.columns:
            df["size_b"] = df["model"].map(size_map)

    print("All results loaded.")

    # ── Figure 1: Accuracy scaling ────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
    for ax, ds in zip(axes, DATASETS):
        subset = acc_df[acc_df.dataset == ds].sort_values("size_b")
        for mode in ["direct", "cot"]:
            m_data = subset[subset['mode'] == mode]
            ax.plot(m_data["size_b"], m_data["accuracy"], color=COLORS[mode], marker=MARKERS[mode], label=mode.capitalize())
        ax.set_title(DATASET_LABELS[ds])
        ax.set_xlabel("Model size (B params)")
        ax.set_ylabel("Accuracy")
        ax.set_ylim(0, 1)
        ax.axvline(x=3, color="gray", linestyle="--", alpha=0.4, label="3B threshold")
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
    fig.suptitle("Figure 1: Accuracy (Direct vs CoT) across Model Sizes", fontsize=13)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "fig1_accuracy_scaling.png", bbox_inches="tight")
    plt.show()
    print("fig1 saved")

    # ── Figure 2: ECE scaling ──────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, ds in zip(axes, ["gsm8k", "truthfulqa"]):
        subset = ece_df[ece_df.dataset == ds].sort_values("size_b")
        for mode in ["direct", "cot"]:
            m_data = subset[subset['mode'] == mode]
            ax.plot(m_data["size_b"], m_data["ece"], color=COLORS[mode], marker=MARKERS[mode], label=mode.capitalize())
        ax.set_title(DATASET_LABELS[ds])
        ax.set_xlabel("Model size (B params)")
        ax.set_ylabel("ECE (lower = better calibrated)")
        ax.axvline(x=3, color="gray", linestyle="--", alpha=0.4, label="3B threshold")
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
    fig.suptitle("Figure 2: Expected Calibration Error (Direct vs CoT)", fontsize=13)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "fig2_ece_scaling.png", bbox_inches="tight")
    plt.show()
    print("fig2 saved")

    # ── Figure 3: Reliability diagrams ─────────────────────────────────────
    representative = ["Qwen2.5-0.5B", "Qwen2.5-1.5B", "Qwen2.5-3B", "Llama-3.1-8B"]
    fig, axes = plt.subplots(2, 4, figsize=(18, 8))
    for col, model_name in enumerate(representative):
        for row, mode in enumerate(["direct", "cot"]):
            ax = axes[row][col]
            key = f"{model_name}/{mode}/gsm8k"
            bins = bin_data.get(key, [])
            if not bins:
                ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
            else:
                mids  = [b["bin_mid"] for b in bins]
                accs  = [b["bin_acc"] for b in bins]
                ax.bar(mids, accs, width=0.08, alpha=0.6, color=COLORS[mode], label="Accuracy")
                ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Perfect cal.")
            ax.set_xlim(0, 1)
            ax.set_ylim(0, 1)
            ax.set_title(f"{model_name}\n{mode}", fontsize=9)
            if col == 0:
                ax.set_ylabel("Accuracy")
            if row == 1:
                ax.set_xlabel("Confidence")
    fig.suptitle("Figure 3: Reliability Diagrams - GSM8K, 4 representative models", fontsize=13)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "fig3_reliability_diagrams.png", bbox_inches="tight")
    plt.show()
    print("fig3 saved")

    # ── Figure 4: Consistency delta ────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, ds in zip(axes, ["gsm8k", "truthfulqa"]):
        subset = delta_df[delta_df.dataset == ds].sort_values("size_b")
        colors = ["#d62728" if d < 0 else "#2ca02c" for d in subset["delta"]]
        ax.bar(range(len(subset)), subset["delta"], color=colors, alpha=0.8)
        ax.set_xticks(range(len(subset)))
        ax.set_xticklabels(subset["model"], rotation=30, ha="right", fontsize=8)
        ax.axhline(0, color="black", linewidth=0.8)
        ax.set_ylabel("Consistency Delta (CoT - Direct)")
        ax.set_title(f"{DATASET_LABELS[ds]}")
        ax.text(0.02, 0.95, "Green = CoT helps consistency\nRed = CoT hurts consistency",
                transform=ax.transAxes, fontsize=8, va="top")
        ax.grid(True, axis="y", alpha=0.3)
    fig.suptitle("Figure 4: Consistency Delta (CoT - Direct) per Model", fontsize=13)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "fig4_consistency_delta.png", bbox_inches="tight")
    plt.show()
    print("fig4 saved")

    # ── Figure 5: Combined trustworthiness heatmap ─────────────────────────
    models_ordered = sorted(MODEL_NAMES, key=lambda m: size_map[m])

    acc_cot_gsm = acc_df[(acc_df['mode'] == "cot") & (acc_df.dataset == "gsm8k")].set_index("model")["accuracy"]
    acc_dir_gsm = acc_df[(acc_df['mode'] == "direct") & (acc_df.dataset == "gsm8k")].set_index("model")["accuracy"]
    acc_delta_gsm = acc_cot_gsm - acc_dir_gsm

    ece_cot_gsm = ece_df[(ece_df['mode'] == "cot") & (ece_df.dataset == "gsm8k")].set_index("model")["ece"]
    ece_dir_gsm = ece_df[(ece_df['mode'] == "direct") & (ece_df.dataset == "gsm8k")].set_index("model")["ece"]
    ece_delta_gsm = ece_cot_gsm - ece_dir_gsm

    con_delta_gsm = delta_df[delta_df.dataset == "gsm8k"].set_index("model")["delta"]

    heatmap_data = pd.DataFrame({
        "Acc delta\n(CoT-Direct)": acc_delta_gsm,
        "ECE delta\n(CoT-Direct)": ece_delta_gsm,
        "Cons delta\n(CoT-Direct)": con_delta_gsm,
    }).reindex(models_ordered)

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(heatmap_data, annot=True, fmt=".3f", cmap="RdYlGn", center=0, linewidths=0.5, ax=ax)
    ax.set_title("Figure 5: CoT Effect on 3 Trustworthiness Axes\n(Green = CoT helps, Red = CoT hurts)", fontsize=12)
    ax.set_xlabel("Metric")
    ax.set_ylabel("Model (sorted by size, top=smallest)")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "fig5_combined_trustworthiness.png", bbox_inches="tight")
    plt.show()
    print("fig5 saved")

    # ── Hypothesis test ─────────────────────────────────────────────────────
    print("=== HYPOTHESIS TEST ===\n")

    print("H1: CoT worsens calibration (higher ECE) for sub-3B models")
    sub3b = ece_df[ece_df.size_b < 3]
    h1 = (sub3b[sub3b['mode'] == "cot"]["ece"].mean() > sub3b[sub3b['mode'] == "direct"]["ece"].mean())
    print(f"  sub-3B ECE(CoT)={sub3b[sub3b['mode']=='cot']['ece'].mean():.3f}, "
          f"ECE(Direct)={sub3b[sub3b['mode']=='direct']['ece'].mean():.3f}")
    print(f"  H1 CONFIRMED: {h1}\n")

    print("H2: CoT decreases consistency for sub-3B models")
    sub3b_con = delta_df[delta_df.size_b < 3]
    h2 = sub3b_con["delta"].mean() < 0
    print(f"  sub-3B mean consistency delta (CoT-Direct): {sub3b_con['delta'].mean():.3f}")
    print(f"  H2 CONFIRMED: {h2}\n")

    print("H3: Pattern reverses at >= 3B (CoT helps both metrics)")
    above3b = delta_df[delta_df.size_b >= 3]
    ece_above = ece_df[ece_df.size_b >= 3]
    h3a = (ece_above[ece_above['mode'] == "cot"]["ece"].mean() < ece_above[ece_above['mode'] == "direct"]["ece"].mean())
    h3b = above3b["delta"].mean() > 0
    print(f"  >= 3B ECE(CoT) < ECE(Direct): {h3a}")
    print(f"  >= 3B consistency delta > 0: {h3b}")
    print(f"  H3 CONFIRMED: {h3a and h3b}")

    print("\nSTEP 5 COMPLETE - All figures saved to figures/")
    print("\nFiles ready for paper:")
    for f in sorted(Path(FIGURES_DIR).glob("*.png")):
        print(f"  {f.name}")
    mark_stage_done(5)


## Save session

Zips `results/` for download; upload as a new dataset version to resume next session.

In [ ]:
from src.session_save import save
save(P["results"])


## Resume status

In [ ]:
print("\nPipeline checkpoint status:")
for step in range(6):
    print(f"  step{step}: {'DONE' if stage_done(step) else 'pending'}")
print("\nTo resume after a Kaggle timeout:")
print("1. Download the results zip printed above.")
print("2. Save /kaggle/working/light_paper (or just its results/ + checkpoints/) as a new dataset version.")
print("3. Attach that dataset next session and click Run All -- it restores the project and skips done stages.")
